# Day 03 Assignment — Joins & Windows

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

In [ ]:
sales = spark.createDataFrame(
    [
        ('c1', '2024-01-01', 100),
        ('c1', '2024-01-02', 150),
        ('c1', '2024-01-02', 150),
        ('c2', '2024-01-01', 80),
        ('c2', '2024-01-03', 200),
        ('c3', '2024-01-01', 50),
    ],
    ['customer_id', 'sale_date', 'amount'],
)
dim = spark.createDataFrame(
    [('c1', 'East'), ('c2', 'West'), ('c3', 'East'), ('c4', 'North')],
    ['customer_id', 'region'],
)

# A1. Left join + anti join for customers with no sales.

In [ ]:
customers_with_sales = dim.join(
    sales,
    on="customer_id",
    how="left"
)

customers_with_sales.show()

no_sales = dim.join(
    sales,
    on="customer_id",
    how="left_anti"
)

no_sales.show()

+-----------+------+----------+------+
|customer_id|region| sale_date|amount|
+-----------+------+----------+------+
|         c2|  West|2024-01-03|   200|
|         c2|  West|2024-01-01|    80|
|         c1|  East|2024-01-02|   150|
|         c1|  East|2024-01-02|   150|
|         c1|  East|2024-01-01|   100|
|         c3|  East|2024-01-01|    50|
|         c4| North|      NULL|  NULL|
+-----------+------+----------+------+

+-----------+------+
|customer_id|region|
+-----------+------+
|         c4| North|
+-----------+------+



# A2. Totals by region.


In [ ]:
region_sales = (
    dim
    .join(sales, on="customer_id", how="left")
    .groupBy("region")
    .agg(
        F.coalesce(F.sum("amount"), F.lit(0)).alias("total_sales")
    )
)

region_sales.show()

+------+-----------+
|region|total_sales|
+------+-----------+
|  West|        280|
| North|          0|
|  East|        450|
+------+-----------+



# A3. lag previous amount + day-over-day change.

In [ ]:
from numpy import record
# 1st we need the Calculate separately for every customer
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy("sale_date")
)

# then you have process the customer records separately
sales_lag = sales.withColumn(
    "previous_amount",
    F.lag("amount").over(window_spec)
)

sales_lag.show()

# then you can calculate the day-over-day change
sales_lag = sales_lag.withColumn(
    "day_over_day_change",
    F.col("amount") - F.col("previous_amount")
)

sales_lag.show()

+-----------+----------+------+---------------+
|customer_id| sale_date|amount|previous_amount|
+-----------+----------+------+---------------+
|         c1|2024-01-01|   100|           NULL|
|         c1|2024-01-02|   150|            100|
|         c1|2024-01-02|   150|            150|
|         c2|2024-01-01|    80|           NULL|
|         c2|2024-01-03|   200|             80|
|         c3|2024-01-01|    50|           NULL|
+-----------+----------+------+---------------+

+-----------+----------+------+---------------+-------------------+
|customer_id| sale_date|amount|previous_amount|day_over_day_change|
+-----------+----------+------+---------------+-------------------+
|         c1|2024-01-01|   100|           NULL|               NULL|
|         c1|2024-01-02|   150|            100|                 50|
|         c1|2024-01-02|   150|            150|                  0|
|         c2|2024-01-01|    80|           NULL|               NULL|
|         c2|2024-01-03|   200|            

# A4. Deduplicate Customer + Date Using ROW_NUMBER

In [ ]:
# first you have to define the window
dedup_window = (
    Window
    .partitionBy("customer_id", "sale_date")
    .orderBy(F.col("amount").desc())
)

# All orders the apply Row_number()
sales_dedup = (
    sales
    .withColumn(
        "rn",
        F.row_number().over(dedup_window)
    )
)
sales_dedup.show()

# keep only return = 1
sales_dedup = (
    sales
    .withColumn(
        "rn",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

sales_dedup.show()

+-----------+----------+------+---+
|customer_id| sale_date|amount| rn|
+-----------+----------+------+---+
|         c1|2024-01-01|   100|  1|
|         c1|2024-01-02|   150|  1|
|         c1|2024-01-02|   150|  2|
|         c2|2024-01-01|    80|  1|
|         c2|2024-01-03|   200|  1|
|         c3|2024-01-01|    50|  1|
+-----------+----------+------+---+

+-----------+----------+------+
|customer_id| sale_date|amount|
+-----------+----------+------+
|         c1|2024-01-01|   100|
|         c1|2024-01-02|   150|
|         c2|2024-01-01|    80|
|         c2|2024-01-03|   200|
|         c3|2024-01-01|    50|
+-----------+----------+------+



# A5. When broadcast dim?

                 Small dim
                    |
          +---------+---------+
          |         |         |
       Executor  Executor  Executor
          |         |         |
       Sales      Sales      Sales

In [ ]:
from pyspark.sql.functions import broadcast

result = sales.join(
    broadcast(dim),
    on="customer_id",
    how="inner"
)

result.show()

+-----------+----------+------+------+
|customer_id| sale_date|amount|region|
+-----------+----------+------+------+
|         c1|2024-01-01|   100|  East|
|         c1|2024-01-02|   150|  East|
|         c1|2024-01-02|   150|  East|
|         c2|2024-01-01|    80|  West|
|         c2|2024-01-03|   200|  West|
|         c3|2024-01-01|    50|  East|
+-----------+----------+------+------+

